In [82]:
import pandas as pd
from pathlib import Path

# Configuração de caminhos
BASE_LONG = Path.cwd().parent / 'dados' / 'processado' / 'base_longitudinal_rr_2019_2025.parquet'
df = pd.read_parquet(BASE_LONG)

# 1. Separar a base de inferência futura (2025)
# O alvo ABANDONO_XXX_T1 (2026) não existe ainda. Usaremos isso apenas no final do projeto para prever o risco futuro.
df_inferencia = df[df['ANO_ALVO'].isna()].copy()

# 2. Separar a base de treino e validação (2019-2024)
df_treino_teste = df[df['ANO_ALVO'].notna()].copy()

# 3. Remover 2019 do treino (Tratamento de Lags)
# Como a base começa em 2019, as colunas '_LAG1' e '_DELTA1' são nulas. 
# Manter 2019 forçaria o modelo a imputar valores falsos para o passado de todas as escolas.
df_treino_teste = df_treino_teste[df_treino_teste['ANO'] > 2019].copy()

# 4. Definir colunas base de identificação que vão para todos os recortes
cols_base = [
    'ANO', 'ANO_ALVO', 'CO_ENTIDADE', 'NO_ENTIDADE', 'CO_MUNICIPIO', 
    'LOCALIZACAO', 'DEPENDENCIA', 'REDE_PUBLICA'
]

# 5. Criar dataset específico para Anos Finais (Foco 9º ano)
# Filtra apenas linhas que possuem a variável alvo preenchida para os Anos Finais
df_af = df_treino_teste[df_treino_teste['ABANDONO_FUN_AF_T1'].notna()].copy()

# Seleciona as colunas de identificação + todas que contêm '_AF' ou '_FUN' (ignorando '_AI' que são Anos Iniciais)
cols_af = [c for c in df.columns if ('_AF' in c or '_FUN' in c) and ('_AI' not in c)]
df_af = df_af[cols_base + cols_af]

# 6. Criar dataset específico para Ensino Médio (1º ao 3º ano)
# Filtra apenas linhas que possuem a variável alvo preenchida para o Ensino Médio
df_med = df_treino_teste[df_treino_teste['ABANDONO_MED_T1'].notna()].copy()

# Seleciona as colunas de identificação + todas que contêm '_MED'
cols_med = [c for c in df.columns if '_MED' in c]
df_med = df_med[cols_base + cols_med]

print(f"Total original: {df.shape[0]} linhas\n")
print(f"Base Treino/Teste - Anos Finais (2020-2024): {df_af.shape[0]} linhas, {df_af.shape[1]} colunas")
print(f"Base Treino/Teste - Ensino Médio (2020-2024): {df_med.shape[0]} linhas, {df_med.shape[1]} colunas")
print(f"Base de Inferência para uso futuro (2025): {df_inferencia.shape[0]} linhas")

#print(df_af.columns)
#print(df_med.columns)
#print(df.shape)

Total original: 6054 linhas

Base Treino/Teste - Anos Finais (2020-2024): 1245 linhas, 32 colunas
Base Treino/Teste - Ensino Médio (2020-2024): 873 linhas, 25 colunas
Base de Inferência para uso futuro (2025): 960 linhas


In [83]:

# célula 2
import pandas as pd
from sklearn.impute import SimpleImputer

print("--- ETAPA 2: TRATAMENTO DE NULOS E MARCAÇÃO DA PANDEMIA ---\n")

def limpar_e_imputar(df, target_col):
    # 1. Remover linhas onde não temos o gabarito (Target nulo)
    df_clean = df.dropna(subset=[target_col]).copy()
    linhas_removidas = len(df) - len(df_clean)
    
    # 2. Criar Flag de Pandemia (2020 e 2021) com base na coluna 'ANO'
    if 'ANO' in df_clean.columns:
        df_clean['FLAG_PANDEMIA'] = df_clean['ANO'].apply(lambda x: 1 if x in [2020, 2021] else 0)
    elif 'NU_ANO_CENSO' in df_clean.columns:
        df_clean['FLAG_PANDEMIA'] = df_clean['NU_ANO_CENSO'].apply(lambda x: 1 if x in [2020, 2021] else 0)
    """# Teste Pé de meia
    if 'ANO' in df_clean.columns:
        df_clean['PE_DE_MEIA_MED'] = df_clean['ANO'].apply(lambda x: 1 if x in [2024, 2025] else 0) * (df_clean['_MED' in df_clean.columns])
    elif 'NU_ANO_CENSO' in df_clean.columns:
        df_clean['PE_DE_MEIA'] = df_clean['NU_ANO_CENSO'].apply(lambda x: 1 if x in [2024, 2025] else 0)"""
        
    # 3. Separar colunas que não devem ser imputadas (identificadores e metadados)
    cols_ignorar = [
        'ANO', 'ANO_ALVO', 'CO_ENTIDADE', 'NO_ENTIDADE', 'CO_MUNICIPIO', 
        'LOCALIZACAO', 'DEPENDENCIA', 'REDE_PUBLICA', target_col, 'FLAG_PANDEMIA'
    ]
    cols_imputar = [c for c in df_clean.columns if c not in cols_ignorar and df_clean[c].dtype in ['float64', 'int64']]
    
    # 4. Remover colunas com mais de 40% de valores nulos
    limite_nulos = 0.001
    nulos_pct = df_clean[cols_imputar].isnull().mean()
    cols_remover = nulos_pct[nulos_pct > limite_nulos].index.tolist()
    
    df_clean = df_clean.drop(columns=cols_remover)
    cols_imputar = [c for c in cols_imputar if c not in cols_remover]
    
    # 5. Imputar nulos restantes usando a Mediana
    if len(cols_imputar) > 0:
        imputer = SimpleImputer(strategy='median')
        df_clean[cols_imputar] = imputer.fit_transform(df_clean[cols_imputar])
    
    # Relatório do processamento
    print(f"Linhas removidas por falta de target: {linhas_removidas}")
    print(f"Colunas removidas (>{limite_nulos*100}% nulos): {len(cols_remover)}")
    if len(cols_remover) > 0:
        print(f" -> {cols_remover}")
    
    return df_clean

# Processamento usando os nomes corretos vindos da Célula 1
print("Processando Anos Finais...")
df_af_clean = limpar_e_imputar(df_af, target_col='ABANDONO_FUN_AF_T1')

print("\nProcessando Ensino Médio...")
df_em_clean = limpar_e_imputar(df_med, target_col='ABANDONO_MED_T1')

print(f"\nShape final Anos Finais: {df_af_clean.shape}")
print(f"Shape final Ensino Médio: {df_em_clean.shape}")


--- ETAPA 2: TRATAMENTO DE NULOS E MARCAÇÃO DA PANDEMIA ---

Processando Anos Finais...
Linhas removidas por falta de target: 0
Colunas removidas (>0.1% nulos): 0

Processando Ensino Médio...
Linhas removidas por falta de target: 0
Colunas removidas (>0.1% nulos): 0

Shape final Anos Finais: (1245, 33)
Shape final Ensino Médio: (873, 26)


In [84]:
print("--- ETAPA 3: SPLIT TEMPORAL (TREINO E TESTE) ---\n")

def separar_treino_teste_temporal(df, target_col, ano_corte=2024):
    # Separar os dados pelo ano (usando 'ANO' em vez de 'NU_ANO_CENSO')
    df_treino = df[df['ANO'] < ano_corte].copy()
    df_teste = df[df['ANO'] == ano_corte].copy()
    
    # Definir as colunas de identificadores e metadados que não devem ir para o modelo

    # Na função separar_treino_teste_temporal da Célula 3:
    cols_ignorar = [
        'ANO', 'ANO_ALVO', 'CO_ENTIDADE', 'NO_ENTIDADE', 'CO_MUNICIPIO',
        'NU_ANO_CENSO', target_col,
        'ABANDONO_FUN_T1', 'REPROVACAO_FUN_T1', 'APROVACAO_FUN_T1'
    ]
    features = [c for c in df.columns if c not in cols_ignorar]
    
    # Separar X (features) e y (gabarito) para treino
    X_train = df_treino[features]
    y_train = df_treino[target_col]
    
    # Separar X e y para teste
    X_test = df_teste[features]
    y_test = df_teste[target_col]
    
    print(f"Período de Treino: {df_treino['ANO'].min()} a {df_treino['ANO'].max()} -> {X_train.shape[0]} amostras")
    print(f"Período de Teste : {ano_corte} -> {X_test.shape[0]} amostras")
    print(f"Quantidade de features: {len(features)}\n")
    
    return X_train, X_test, y_train, y_test, features

print("Separando base de Anos Finais...")
# Corrigido o nome do target para 'ABANDONO_FUN_AF_T1'
X_train_af, X_test_af, y_train_af, y_test_af, features_af = separar_treino_teste_temporal(
    df_af_clean, target_col='ABANDONO_FUN_AF_T1', ano_corte=2024
)

print("Separando base de Ensino Médio...")
X_train_em, X_test_em, y_train_em, y_test_em, features_em = separar_treino_teste_temporal(
    df_em_clean, target_col='ABANDONO_MED_T1', ano_corte=2024
)
#print(features_af)
#print(features_em)

--- ETAPA 3: SPLIT TEMPORAL (TREINO E TESTE) ---

Separando base de Anos Finais...
Período de Treino: 2020 a 2023 -> 989 amostras
Período de Teste : 2024 -> 256 amostras
Quantidade de features: 26

Separando base de Ensino Médio...
Período de Treino: 2020 a 2023 -> 692 amostras
Período de Teste : 2024 -> 181 amostras
Quantidade de features: 20



In [85]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("--- ETAPA 4: TREINAMENTO E AVALIAÇÃO DOS MODELOS ---\n")

def treinar_e_avaliar(X_train, X_test, y_train, y_test, nome_base):
    print(f"=== Resultados para: {nome_base} ===")
    
    # Garantir apenas colunas numéricas e preencher eventuais NaNs residuais
    X_train_num = X_train.select_dtypes(include=[np.number]).fillna(0)
    X_test_num = X_test.select_dtypes(include=[np.number]).fillna(0)
    
    # Dicionário de modelos
    modelos = {
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
    }
    
    melhor_modelo_obj = None
    melhor_r2 = -float('inf')
    nome_melhor = ""
    
    for nome, modelo in modelos.items():
        # Treinamento
        modelo.fit(X_train_num, y_train)
        
        # Inferência na base de teste (2024)
        y_pred = modelo.predict(X_test_num)
        
        # Cálculo das métricas
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        print(f"[{nome}]")
        print(f"   MAE  (Erro Médio Absoluto) : {mae:.2f}%")
        print(f"   RMSE (Raiz do Erro Quad.)  : {rmse:.2f}%")
        print(f"   R²   (Coef. Determinação)  : {r2:.4f}\n")
        
        # Rastreando o melhor modelo com base no R²
        if r2 > melhor_r2:
            melhor_r2 = r2
            melhor_modelo_obj = modelo
            nome_melhor = nome
            
    # Extração de Feature Importance do melhor modelo
    importancias = melhor_modelo_obj.feature_importances_
    df_importancia = pd.DataFrame({
        'Feature': X_train_num.columns,
        'Importancia': importancias
    }).sort_values(by='Importancia', ascending=False)
    
    print(f"-> Melhor modelo escolhido: {nome_melhor}")
    print("-> Top 5 variáveis com maior peso na predição:")
    print(df_importancia.head(5).to_string(index=False))
    print("-" * 50 + "\n")
    
    return melhor_modelo_obj

# Executar treinamento para as duas bases
modelo_final_af = treinar_e_avaliar(X_train_af, X_test_af, y_train_af, y_test_af, "Anos Finais")
modelo_final_em = treinar_e_avaliar(X_train_em, X_test_em, y_train_em, y_test_em, "Ensino Médio")

--- ETAPA 4: TREINAMENTO E AVALIAÇÃO DOS MODELOS ---

=== Resultados para: Anos Finais ===
[Random Forest]
   MAE  (Erro Médio Absoluto) : 2.58%
   RMSE (Raiz do Erro Quad.)  : 3.72%
   R²   (Coef. Determinação)  : 0.1843

[Gradient Boosting]
   MAE  (Erro Médio Absoluto) : 2.46%
   RMSE (Raiz do Erro Quad.)  : 3.69%
   R²   (Coef. Determinação)  : 0.2011

-> Melhor modelo escolhido: Gradient Boosting
-> Top 5 variáveis com maior peso na predição:
          Feature  Importancia
    TDI_FUN_TOTAL     0.200766
       TDI_FUN_AF     0.102163
     ABANDONO_FUN     0.091239
REPROVACAO_FUN_AF     0.083763
TDI_FUN_AF_DELTA1     0.051138
--------------------------------------------------

=== Resultados para: Ensino Médio ===
[Random Forest]
   MAE  (Erro Médio Absoluto) : 4.11%
   RMSE (Raiz do Erro Quad.)  : 6.13%
   R²   (Coef. Determinação)  : 0.1214

[Gradient Boosting]
   MAE  (Erro Médio Absoluto) : 4.13%
   RMSE (Raiz do Erro Quad.)  : 6.23%
   R²   (Coef. Determinação)  : 0.0907

-> M